# 02 — Regime Detection
3-tier regime classification: fast (1s), medium (1m), slow (15m HMM).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from hmmlearn.hmm import GaussianHMM
from config import *
from utils import rolling_variance_ratio, rolling_hurst, rolling_autocorrelation

DATA_DIR = Path('../data')

In [ ]:
features = pd.read_parquet(DATA_DIR / 'features.parquet')
returns = features['log_return_1m'].dropna()
print(f'Loaded {len(features):,} bars')

## Tier 1: Fast Signals (1-second update)

In [ ]:
# These operate on 1-second data in production.
# Here we approximate from 1-min bars for offline analysis.

# Microprice divergence proxy: buy_ratio deviation from 0.5
if 'buy_ratio' in features.columns:
    features['microprice_div'] = features['buy_ratio'] - 0.5

# Order book imbalance velocity proxy: rate of change of buy_ratio
if 'buy_ratio' in features.columns:
    features['ob_imbalance_velocity'] = features['buy_ratio'].diff()

# Trade arrival asymmetry: rolling 10-bar buy/sell count ratio
if 'buy_ratio' in features.columns:
    features['trade_arrival_asymmetry'] = features['buy_ratio'].rolling(10, min_periods=1).mean() - 0.5

print('Tier 1 signals computed (proxy from 1-min bars)')

## Tier 2: Medium Signals (1-minute update)

In [ ]:
features['variance_ratio'] = rolling_variance_ratio(returns)
features['hurst'] = rolling_hurst(returns)
features['autocorr_lag1'] = rolling_autocorrelation(returns)

# VPIN rate of change (VPIN computed in features already)
if 'vpin' in features.columns:
    features['vpin_roc_5m'] = features['vpin'].diff(5)

print('Tier 2 signals computed')
print(features[['variance_ratio', 'hurst', 'autocorr_lag1']].describe().round(4))

## Tier 3: HMM (15-minute update)

In [ ]:
# Observable vector for HMM: [1-min return, rv_5m, tofi_5m, basis_momentum_5m]
hmm_cols = ['log_return_1m']
for c in ['rv_5m', 'spot_tofi_5m', 'perp_tofi_5m', 'basis_momentum_5m']:
    if c in features.columns:
        hmm_cols.append(c)

hmm_data = features[hmm_cols].dropna()
print(f'HMM observable dimensions: {len(hmm_cols)}')
print(f'HMM training samples: {len(hmm_data):,}')

In [ ]:
# Fit 4-state Gaussian HMM (Student-t not directly supported by hmmlearn;
# Gaussian is a reasonable approximation for regime identification)
hmm = GaussianHMM(
    n_components=HMM_N_STATES,
    covariance_type='full',
    n_iter=100,
    random_state=42
)

X = hmm_data.values
hmm.fit(X)
print(f'HMM converged: {hmm.monitor_.converged}')
print(f'Log-likelihood: {hmm.score(X):.2f}')

In [ ]:
# Decode states and compute posteriors
state_sequence = hmm.predict(X)
posteriors = hmm.predict_proba(X)

# Label states by mean return: highest mean = Bull, lowest = Bear, etc.
state_means = pd.DataFrame(X, columns=hmm_cols).groupby(state_sequence)['log_return_1m'].mean()
state_order = state_means.sort_values().index.tolist()
state_labels = {state_order[0]: 'Bear', state_order[-1]: 'Bull'}

# Middle two: lower vol = Calm, higher vol = Transition
middle = [s for s in range(HMM_N_STATES) if s not in [state_order[0], state_order[-1]]]
state_vols = pd.DataFrame(X, columns=hmm_cols).groupby(state_sequence)['log_return_1m'].std()
if len(middle) == 2:
    if state_vols[middle[0]] < state_vols[middle[1]]:
        state_labels[middle[0]] = 'Calm'
        state_labels[middle[1]] = 'Transition'
    else:
        state_labels[middle[0]] = 'Transition'
        state_labels[middle[1]] = 'Calm'
elif len(middle) == 1:
    state_labels[middle[0]] = 'Calm'

# Map to feature DataFrame
regime_df = pd.DataFrame(index=hmm_data.index)
regime_df['hmm_state'] = [state_labels.get(s, f'State_{s}') for s in state_sequence]
for i in range(HMM_N_STATES):
    label = state_labels.get(i, f'State_{i}')
    regime_df[f'hmm_prob_{label.lower()}'] = posteriors[:, i]

print('State distribution:')
print(regime_df['hmm_state'].value_counts(normalize=True).round(3))

## Composite Regime Score

In [ ]:
# Map each tier-2 signal to [-1, +1]: negative = mean-reverting, positive = trending
aligned = features.reindex(hmm_data.index)

vr = aligned['variance_ratio'].clip(0, 2)
vr_signal = (vr - 1.0)  # VR<1 → negative (mean-reverting), VR>1 → positive (trending)

h = aligned['hurst'].clip(0, 1)
hurst_signal = (h - 0.5) * 2  # H<0.5 → negative, H>0.5 → positive

# HMM trend probability: P(bull) + P(bear) - P(calm) - P(transition)
hmm_trend = 0.0
for col in regime_df.columns:
    if 'bull' in col or 'bear' in col:
        hmm_trend = hmm_trend + regime_df[col]
    elif 'calm' in col:
        hmm_trend = hmm_trend - regime_df[col]

ac = aligned['autocorr_lag1'].clip(-1, 1)

# Composite (equal weights as starting point)
regime_score = 0.25 * vr_signal + 0.25 * hurst_signal + 0.25 * hmm_trend + 0.25 * ac
regime_score = regime_score.clip(-1, 1)
regime_df['regime_score'] = regime_score

print('Regime score distribution:')
print(regime_score.describe().round(4))

In [ ]:
# Quoting behavior mapping
def regime_to_behavior(score):
    if score < -0.3:
        return 'tight_symmetric'
    elif score < 0.3:
        return 'moderate'
    elif score < 0.6:
        return 'wide_skewed'
    else:
        return 'cease_or_directional'

regime_df['quoting_behavior'] = regime_score.apply(regime_to_behavior)
print('Quoting behavior distribution:')
print(regime_df['quoting_behavior'].value_counts(normalize=True).round(3))

In [ ]:
# Save regime data
regime_df.to_parquet(DATA_DIR / 'regimes.parquet')
print(f'Saved {len(regime_df):,} regime labels')

## Validation

In [ ]:
# Transition matrix
states = regime_df['hmm_state']
transitions = pd.crosstab(states, states.shift(-1), normalize='index')
print('Transition matrix:')
print(transitions.round(3))

# Average regime duration
runs = (states != states.shift()).cumsum()
durations = states.groupby(runs).agg(['first', 'count'])
avg_duration = durations.groupby('first')['count'].mean()
print('\nAvg regime duration (minutes):')
print(avg_duration.round(1))

In [ ]:
# Time-series plot: price with regime coloring
price = features['spot_close'].reindex(hmm_data.index)
if len(price) > 0:
    sample = price.iloc[-2000:]  # last ~33 hours
    sample_states = regime_df['hmm_state'].reindex(sample.index)

    colors = {'Bull': 'green', 'Bear': 'red', 'Calm': 'blue', 'Transition': 'orange'}
    fig, ax = plt.subplots(figsize=(14, 4))
    for state, color in colors.items():
        mask = sample_states == state
        if mask.any():
            ax.scatter(sample.index[mask], sample[mask], c=color, s=1, label=state, alpha=0.7)
    ax.set_title('Price colored by HMM regime')
    ax.legend(markerscale=5)
    plt.tight_layout()
    plt.show()